In [ ]:
from sqlite3 import connect
from glob import glob
from collections import Counter
from os import remove
from functools import reduce
import numpy as np
import torch
from torch.nn import *
class Res(Sequential):
    def forward(s, x):
        return super().forward(x)+x
class At(MultiheadAttention):
    def forward(s, x):
        return super().forward(x, x, x, attn_mask=Transformer.generate_square_subsequent_mask(x.size(-2),dtype=torch.half,device=x.device), is_causal=True)[0]
class Swiglu(Linear):
    def __init__(s, i, o=0):
        super().__init__(i, 2*(o or i//6*16))
        s._ = Linear(o or i//6*16, i)
    def forward(s, x):
        x,_ = super().forward(x).chunk(2,-1)
        return s._(x*functional.silu(_))
class REnCoder(Module):
    def __init__(s, o, *i):
        super().__init__()
        s.i = len(i)
        _ = 0
        for j,i in enumerate(i):
            if isinstance(i,(int,float)):
                setattr(s, str(j), (Embedding if isinstance(i,int)else EmbeddingBag)(int(i),o,scale_grad_by_freq=True))
            else:
                _ += 1
        s._ = Linear(_, o)
    def forward(s, x):
        assert x.size(-1)==s.i, (x.size(-1),s.i)
        o = 0
        _ = []
        for i in range(x.size(-1)):
            if isinstance(m:=getattr(s,str(i),None), Embedding):
                o += m(x[...,i])
            elif isinstance(m, EmbeddingBag):
                o += (1<<torch.arange(m.num_embeddings,dtype=x.dtype,device=x.device)&x[...,i,None]).bool().to(m.weight.dtype)@m.weight
            else:
                _ += [i]
        return o + s._(x[...,_].to(s._.weight.dtype))
def Try(f, default=''):
    try:
        return f()
    except FileNotFoundError:
        pass
    except Exception as _:
        print(_)
    return default
RETRAIN = False
try:
    MAP = dict(map(reversed,enumerate(MAP_:=np.fromfile('model3aa.i','i').tolist())))
except FileNotFoundError as _:
    print(_)
    RETRAIN = True
    try:
        remove('model3aa.pk')
    except FileNotFoundError:
        pass
    MAP = {}
SET = {}
CARDS = {id:(
    alias or id,
    ot-(ot>0x4)*0x4,
    *(reduce(int.__or__,((i<=(_:=setcode>>j*16&0xfff)<i+8 and 1<<(_-i))|(i==57 and setcode<<j*16&0xf000000000000000)for j in range(4)))for i in range(1,64,8)),
    type,
    max(atk,0),
    0 if type&0x4000000 else max(Def,0),
    level&15,
    race.bit_length(),
    attribute.bit_length(),
    category%(1<<32),
    level>>24,
    Def-(Def>0x10)*0x10 if type&0x4000000 else 0,
    len(name),
    len(desc),
    len(_:=Try(lambda:open(f'script/c{alias or id}.lua').read(),'')),
    sum((k in j)<<i for i,j,k in(
        (0,name,'- 星者魔生之战王兽灵人神天机士龙黑的·'),
        (20,desc,'手把·攻「表方时有个那外为次墓在地坏殊从用'),
        (41,_,'SetValue LOCATION_GRAVE LOCATION_HAND SetReset false Card GetCount IsType POS_FACEUP SetCost SetCountLimit IsRelateToEffect IsLocation EVENT_FREE_CHAIN not IsFaceup true tc target IsExistingMatchingCard EFFECT_TYPE_FIELD EFFECT_TYPE_SINGLE IsSetCard'.split())
    )for i,k in enumerate(k,i))
)for id,ot,alias,setcode,type,atk,Def,level,race,attribute,category,name,desc in connect('cards.cdb').execute('select datas.*,name,desc from datas join texts on datas.id=texts.id')}
DECK = [
    torch.tensor(list(Counter(i).items()),dtype=torch.short,device='cuda')
    for i in glob('deck/**/**.ydk',recursive=True,include_hidden=True)
    for i in[[
        MAP.setdefault(CARDS[int(i)][0],len(MAP))
        for i in open(i).read().split('!')[0].split()
        if i.isdigit()and int(i)in(RETRAIN and CARDS or MAP)
    ]]
    if len(i)>=40
]
CARD = torch.tensor([CARDS[i][1:]for i in MAP],dtype=torch.uint64,device='cuda').view(torch.int64)
np.array(list(MAP),'i').tofile('model3aa.i')

In [ ]:
try:
    m,o,s = torch.load('model3aa.pk','cuda',weights_only=False)
except FileNotFoundError as _:
    print(_)
    m = 256
    m = torch.compile(Sequential(
        REnCoder(m,int, 3., 64.,64.,64.,64.,64.,64.,64.,64., 27., int, int, int, 27, 8, 32., int, 8., int, int, int, 64.),
        *(Sequential(
            Res(
                LayerNorm(m),
                At(m, 16, batch_first=True),
            ),
            Res(
                LayerNorm(m),
                Swiglu(m),
            ),
        )for _ in range(5)),
        Linear(m, len(MAP))
    ).cuda())
    o = torch.optim.Adam(m.parameters(), .0008)
    s = torch.amp.GradScaler()    

In [ ]:
while True:
    L = 0
    for _ in range(10000):
        o.zero_grad()
        a = [DECK[i][torch.randperm(len(DECK[i]),dtype=torch.int,device='cuda')]for i in torch.randint(len(DECK),[8],dtype=torch.short)]
        x = torch.stack([torch.constant_pad_nd(torch.cat([i,CARD[i[:,0].int()]],1),(0,0,0,max(map(len,a))-len(i)))for i in a])
        with torch.amp.autocast('cuda', torch.float16):
            y = m(x[:,:-1,1:]).log_softmax(2)
            t = torch.full_like(y,1e-6,dtype=torch.half).scatter_add_(2,x[:,1:,:1].flip(1),x[:,1:,1:2].half().flip(1)).cumsum_(1).flip(1)
            t /= -t.numel()*t.sum(2,True)
            l = y.ravel()@t.ravel()
        s.scale(l).backward()
        s.step(o)
        s.update()
        L += l.data.float()
    print(L)
    torch.save((m,o,s),'model3aa.pk')